In [26]:
import os
from itertools import islice
from math import ceil
import numpy as np
import odl
from tqdm import tqdm
from skimage.transform import resize
from pydicom.filereader import dcmread
import h5py
import random

In [27]:
MU_WATER = 20
MU_AIR = 0.02
MU_MAX = 3071 * (MU_WATER - MU_AIR) / 1000 + MU_WATER

# Image physical size in meters
MIN_PT = [-0.13, -0.13]
MAX_PT = [0.13, 0.13]

# File containing list of accepted CT scan directories
DIR_LIST_FILE = "ct_scan_my.txt"

DATA_PATH = "../data/images"
TARGET_PATH = "../data/dataset_S2010"

TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

NUM_ANGLES = 512
RECO_IM_SHAPE = (256, 256)

# Image shape for simulation
IM_SHAPE = (500, 500)

# ASTRA implementation to use: "astra_cpu" or "astra_cuda"
IMPL = "astra_cuda"

# Whether to add Poisson noise to the simulated sinograms
ADD_NOISE = False

PHOTONS_PER_PIXEL = 4096

# Number of samples stored in each HDF5 file
NUM_SAMPLES_PER_FILE = 128

In [28]:
os.makedirs(TARGET_PATH, exist_ok=True)

all_files = []

# Load list of directories
with open(DIR_LIST_FILE, "r") as f:
    dir_list = [line.strip() for line in f if line.strip()]

for rel_path in dir_list:
    abs_path = os.path.join(DATA_PATH, rel_path)

    if not os.path.isdir(abs_path):
        print(f"Warning: directory does not exist: {abs_path}")
        continue

    files = os.listdir(abs_path)

    if any(f in ("I10", "I10.dcm") for f in files):
        for f in files:
            if f[0] == "I":
                all_files.append(os.path.join(abs_path, f))

print(f"Found {len(all_files)} files in {DATA_PATH} and its subdirectories.")
print(all_files[:3])

Found 161 files in ../data/images and its subdirectories.
['../data/images/S2010/I500', '../data/images/S2010/I390', '../data/images/S2010/I1140']


In [29]:
random.shuffle(all_files)

n_total = len(all_files)
n_train = int(TRAIN_RATIO * n_total)
n_val = int(VAL_RATIO * n_total)
n_test = n_total - n_train - n_val

file_list = {
    "train": all_files[:n_train],
    "validation": all_files[n_train:n_train + n_val],
    "test": all_files[n_train + n_val:]
}

print("Number of training files:", len(file_list["train"]))
print("Number of validation files:", len(file_list["validation"]))
print("Number of test files:", len(file_list["test"]))

Number of training files: 128
Number of validation files: 16
Number of test files: 17


In [30]:
def lidc_idri_gen(part="train"):
    seed = 0
    if part == "validation":
        seed = 1
    elif part == "test":
        seed = 2
    r = np.random.RandomState(seed)
    for dcm_file in file_list[part]:
        dataset = dcmread(os.path.join(dcm_file))

        # crop to largest rectangle in centered circle
        # array = dataset.pixel_array[75:-75, 75:-75].astype(np.float32).T
        array = dataset.pixel_array.astype(np.float32).T

        # resize image to RECO_IM_SHAPE
        array = resize(array, RECO_IM_SHAPE, order=1)

        # rescale by dicom meta info
        array *= dataset.RescaleSlope
        array += dataset.RescaleIntercept

        # add noise to get continuous values from discrete ones
        array += r.uniform(0.0, 1.0, size=array.shape)

        # convert values
        array *= (MU_WATER - MU_AIR) / 1000
        array += MU_WATER
        array /= MU_MAX
        np.clip(array, 0.0, 1.0, out=array)

        yield array

In [31]:
lidc_idri_gen_len = {p: len(file_list[p]) for p in ["train", "validation", "test"]}

In [32]:
reco_space = odl.uniform_discr(
    min_pt=MIN_PT, max_pt=MAX_PT, shape=RECO_IM_SHAPE, dtype=np.float32
)
space = odl.uniform_discr(
    min_pt=MIN_PT, max_pt=MAX_PT, shape=IM_SHAPE, dtype=np.float64
)

reco_geometry = odl.tomo.parallel_beam_geometry(reco_space, num_angles=NUM_ANGLES)
geometry = odl.tomo.parallel_beam_geometry(
    space, num_angles=NUM_ANGLES, det_shape=reco_geometry.detector.shape
)

reco_ray_trafo = odl.tomo.RayTransform(reco_space, reco_geometry, impl=IMPL)
ray_trafo = odl.tomo.RayTransform(space, geometry, impl=IMPL)

print(ray_trafo.range.shape)
print(reco_space.shape)

rs = np.random.RandomState(3)

(512, 365)
(256, 256)


In [33]:
def forward_fun(im):
    # upsample ground_truth in each dimension
    # before application of forward operator in order to avoid
    # inverse crime
    im_resized = resize(im * MU_MAX, IM_SHAPE, order=1)

    # apply forward operator
    data = ray_trafo(im_resized).asarray()

    data *= -1
    np.exp(data, out=data)
    data *= PHOTONS_PER_PIXEL
    
    return data

In [34]:
for part in ["train", "validation", "test"]:
    gen = lidc_idri_gen(part)
    n_files = ceil(lidc_idri_gen_len[part] / NUM_SAMPLES_PER_FILE)
    for filenumber in tqdm(range(n_files), desc=part):
        obs_filename = os.path.join(
            TARGET_PATH, "observation_{}_{:04d}.hdf5".format(part, filenumber)
        )
        ground_truth_filename = os.path.join(
            TARGET_PATH, "ground_truth_{}_{:04d}.hdf5".format(part, filenumber)
        )
        with h5py.File(obs_filename, "w") as observation_file, h5py.File(
            ground_truth_filename, "w"
        ) as ground_truth_file:
            observation_dataset = observation_file.create_dataset(
                "data",
                shape=(NUM_SAMPLES_PER_FILE,) + ray_trafo.range.shape,
                maxshape=(NUM_SAMPLES_PER_FILE,) + ray_trafo.range.shape,
                dtype=np.float32,
                chunks=True,
            )
            ground_truth_dataset = ground_truth_file.create_dataset(
                "data",
                shape=(NUM_SAMPLES_PER_FILE,) + reco_space.shape,
                maxshape=(NUM_SAMPLES_PER_FILE,) + reco_space.shape,
                dtype=np.float32,
                chunks=True,
            )

            im_buf = [im for im in islice(gen, NUM_SAMPLES_PER_FILE)]
            data_buf = [forward_fun(im) for im in im_buf]

            for i, (im, data) in enumerate(zip(im_buf, data_buf)):
                if ADD_NOISE:
                    data = rs.poisson(data)
                data = data / PHOTONS_PER_PIXEL
                np.maximum(0.1 / PHOTONS_PER_PIXEL, data, out=data)
                np.log(data, out=data)
                data /= -MU_MAX
                observation_dataset[i] = data
                ground_truth_dataset[i] = im

            # resize last file
            if filenumber == n_files - 1:
                observation_dataset.resize(
                    lidc_idri_gen_len[part] - (n_files - 1) * NUM_SAMPLES_PER_FILE,
                    axis=0,
                )
                ground_truth_dataset.resize(
                    lidc_idri_gen_len[part] - (n_files - 1) * NUM_SAMPLES_PER_FILE,
                    axis=0,
                )

test: 100%|██████████| 1/1 [00:00<00:00,  2.85it/s]
